# Practical 13: Data Quality Monitoring Using DuckDB

## 1. Extract the Dataset

In [1]:
from pathlib import Path
import zipfile

zip_path = Path("messy e-commerce data.zip")
extract_folder = Path("messy_ecommerce_data")
extract_folder.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Dataset extracted successfully.")

Dataset extracted successfully.


## 2. Load the Datasets

In [2]:
import pandas as pd

customers = pd.read_csv("messy_ecommerce_data/customers.csv")
transactions = pd.read_csv("messy_ecommerce_data/transactions.csv")

print("Customers:", customers.shape)
print("Transactions:", transactions.shape)

display(customers.head())
display(transactions.head())

Customers: (1010, 9)
Transactions: (8199, 10)


,customer_id,name,age,gender,state,signup_date,email,phone_number,subscribe
0,CUST0001,Gandi,26.0,Male,Sulawesi Utara,2025-05-05,gandi_96@example.com,8422757160,No
1,CUST0002,Cecep,49.0,Male,Aceh,2025-06-02,cecep467@example.com,88171122035,No
2,CUST0003,Artawan,26.0,Male,Jambi,2025-02-11,wahyuninovi@example.net,8692932468,Yes
3,CUST0004,Mulya,61.0,Male,Nusa Tenggara Timur,2025-02-19,nnapitupulu@example.com,8515735210,Yes
4,CUST0005,Taufik,43.0,Male,Kepulauan Riau,2025-08-04,taufik_87@example.com,81034538051,Yes


,transaction_id,customer_id,transaction_date,product_id,quantity,unit_price,payment_method,discount_applied,transaction_status,review_text
0,TX_0000003568,CUST0429,2024-12-29,WZ-802,30,448000,Credit Card,6.0,Completed,NaN
1,TX_0000001323,CUST0165,2025-09-20,LE-969,31,326000,E-wallet,7.0,Completed,NaN
2,TX_0000007976,CUST0969,2025-09-19,CR-562,22,326000,E-wallet,8.0,Completed,NaN
3,TX_0000002408,CUST0293,2025-04-23,BL-316,38,418000,Credit Card,42.0,Completed,NaN
4,TX_0000004435,CUST0537,2025-08-28,KL-203,3,15000,E-wallet,27.0,Completed,NaN


## 3. Install and Import DuckDB

In [5]:
import sys
!{sys.executable} -m pip install duckdb


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: C:\Users\Divya\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## 4. Create DuckDB Tables

In [6]:
import duckdb

con = duckdb.connect("practical13.duckdb")

con.register("customers_df", customers)
con.register("transactions_df", transactions)

con.execute("""
CREATE OR REPLACE TABLE customers AS
SELECT * FROM customers_df
""")

con.execute("""
CREATE OR REPLACE TABLE transactions AS
SELECT * FROM transactions_df
""")

print("Customers and transactions tables created in DuckDB.")

Customers and transactions tables created in DuckDB.


## 5. Detect NULL Values

In [7]:
con.execute("""
SELECT
    COUNT(*) - COUNT(discount_applied) AS null_discount_applied,
    COUNT(*) - COUNT(review_text) AS null_review_text
FROM transactions;
""").df()

,null_discount_applied,null_review_text
0,426,4916


## 6. Detect Duplicate Records

In [8]:
con.execute("""
SELECT
    (SELECT COUNT(*) - COUNT(DISTINCT customer_id)
     FROM customers) AS duplicate_customer_ids,
    (SELECT COUNT(*) - COUNT(DISTINCT transaction_id)
     FROM transactions) AS duplicate_transaction_ids;
""").df()

,duplicate_customer_ids,duplicate_transaction_ids
0,10,0


## 7. Detect Invalid Values

In [11]:
con.execute("""
SELECT
    (SELECT SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END)
     FROM transactions) AS invalid_quantity,

    (SELECT SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END)
     FROM transactions) AS invalid_unit_price,

    (SELECT SUM(CASE WHEN discount_applied < 0 OR discount_applied > 100 THEN 1 ELSE 0 END)
     FROM transactions) AS invalid_discount,

    (SELECT SUM(CASE WHEN age < 18 OR age > 100 THEN 1 ELSE 0 END)
     FROM customers) AS invalid_age
""").df()

,invalid_quantity,invalid_unit_price,invalid_discount,invalid_age
0,0.0,0.0,0.0,0.0


## 8. Define Data Quality Rules

### Data Quality Rules

1. `discount_applied` must not be NULL.
2. `customer_id` must be unique.
3. `transaction_id` must be unique.
4. `quantity` must be greater than 0.
5. `unit_price` must be greater than 0.
6. `discount_applied` must be between 0 and 100.
7. Customer age must be between 18 and 100.

## 9. Generate Data Quality Report

In [12]:
quality_report = con.execute("""
SELECT
    'NULL discount_applied' AS quality_check,
    COUNT(*) - COUNT(discount_applied) AS issues,
    CASE
        WHEN COUNT(*) - COUNT(discount_applied) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM transactions

UNION ALL

SELECT
    'Duplicate customer_id',
    COUNT(*) - COUNT(DISTINCT customer_id),
    CASE
        WHEN COUNT(*) - COUNT(DISTINCT customer_id) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END
FROM customers

UNION ALL

SELECT
    'Duplicate transaction_id',
    COUNT(*) - COUNT(DISTINCT transaction_id),
    CASE
        WHEN COUNT(*) - COUNT(DISTINCT transaction_id) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END
FROM transactions

UNION ALL

SELECT
    'Invalid quantity',
    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END),
    CASE
        WHEN SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END
FROM transactions

UNION ALL

SELECT
    'Invalid unit_price',
    SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END),
    CASE
        WHEN SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END
FROM transactions

UNION ALL

SELECT
    'Invalid discount',
    SUM(CASE WHEN discount_applied < 0 OR discount_applied > 100 THEN 1 ELSE 0 END),
    CASE
        WHEN SUM(CASE WHEN discount_applied < 0 OR discount_applied > 100 THEN 1 ELSE 0 END) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END
FROM transactions

UNION ALL

SELECT
    'Invalid customer age',
    SUM(CASE WHEN age < 18 OR age > 100 THEN 1 ELSE 0 END),
    CASE
        WHEN SUM(CASE WHEN age < 18 OR age > 100 THEN 1 ELSE 0 END) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END
FROM customers;
""").df()

quality_report

,quality_check,issues,status
0,NULL discount_applied,426.0,FAIL
1,Duplicate customer_id,10.0,FAIL
2,Duplicate transaction_id,0.0,PASS
3,Invalid quantity,0.0,PASS
4,Invalid unit_price,0.0,PASS
5,Invalid discount,0.0,PASS
6,Invalid customer age,0.0,PASS


## 10. Recommended Corrections

### Recommended Corrections

1. NULL `discount_applied`: Treat missing discount values as 0 if the business rule considers NULL to mean no discount.
2. Duplicate `customer_id`: Remove duplicate customer records while retaining one record for each customer.
3. Duplicate `transaction_id`: No correction required because transaction IDs are unique.
4. Invalid quantity, unit price, discount, and customer age: No correction required because all values passed validation.

## 11. Apply Corrections

In [13]:
con.execute("""
CREATE OR REPLACE TABLE customers_clean AS
SELECT *
FROM customers
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY signup_date
) = 1
""")

con.execute("""
CREATE OR REPLACE TABLE transactions_clean AS
SELECT
    transaction_id,
    customer_id,
    transaction_date,
    product_id,
    quantity,
    unit_price,
    payment_method,
    COALESCE(discount_applied, 0) AS discount_applied,
    transaction_status,
    review_text
FROM transactions
""")

print("Corrections applied successfully.")

Corrections applied successfully.


## 12. Final Validation

In [14]:
con.execute("""
SELECT
    'NULL discount_applied' AS quality_check,
    COUNT(*) - COUNT(discount_applied) AS issues
FROM transactions_clean

UNION ALL

SELECT
    'Duplicate customer_id',
    COUNT(*) - COUNT(DISTINCT customer_id)
FROM customers_clean;
""").df()

,quality_check,issues
0,NULL discount_applied,0
1,Duplicate customer_id,0
